Import the data, load it into a pandas data frame and clean it, in order to converte it into a sqlite database file.

In [ ]:
# import libraries
import pandas as pd
import openpyxl

In [ ]:
# upload file
from google.colab import files
uploaded = files.upload()

Saving students_at_public_universities.xlsx to students_at_public_universities.xlsx


In [ ]:
# load the data into a pandas dataframe
file = "students_at_public_universities.xlsx"
df = pd.read_excel(file, skiprows = 11)

In [ ]:
# drop empty rows and rows filled with metatrash
df.drop([0], inplace = True)
df = df.iloc[:-11]
# drop empty column
df = df.drop(df.columns[0], axis=1)

In [ ]:
# Rename the first two columns for clarity
df.rename(columns={df.columns[0]: "Semester", df.columns[1]: "Field of Study"}, inplace=True)

In [ ]:
# fill semester column accordingly
df["Semester"] = pd.Series(df["Semester"]).ffill()
df

,Semester,Field of Study,University of Vienna,University of Graz,University of Innsbruck,University of Salzburg,Vienna University of Technology,Graz University of Technology,University of Leoben,University of Natural Resources and Life Sciences Vienna,...,University of Leoben.1,University of Natural Resources and Life Sciences Vienna.1,University of Veterinary Medicine Vienna.1,Vienna University of Economics and Business.1,Johannes Kepler University (JKU) Linz.1,University of Klagenfurt.1,Danube University Krems.1,Medical University of Vienna.1,Medical University of Graz.1,Medical University of Innsbruck.1
1,winter semester 2023/24,"05 Natural sciences, mathematics and statistics",6985.81994,2512,2430,797,2412.00012,1376,-,1385.59994,...,-,2133.19997,242.08,-,645.5,78,-,10.6,-,-
2,winter semester 2023/24,051 Biological and related sciences,2263.82,373,515,415,-,189,-,1,...,-,7,185.28,-,355.5,-,-,10.6,-,-
3,winter semester 2023/24,052 Environment,84,329.5,114,-,-,101.5,-,902.6,...,-,1289.2,15.8,-,-,-,-,-,-,-
4,winter semester 2023/24,053 Physical sciences,3229.99994,915.5,1591,201,1307.00012,870.5,-,5.99994,...,-,2.99997,-,-,121,27,-,-,-,-
5,winter semester 2023/24,054 Mathematics and statistics,1318,160,183,90,808,168,-,-,...,-,-,-,-,139,51,-,-,-,-
6,winter semester 2023/24,061 Information and Communication Technologies...,180,-,329,370,3636,1900,-,-,...,-,-,-,-,591,93,-,24,-,-
7,winter semester 2023/24,0611 Computer use,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
8,winter semester 2023/24,0612 Database and network design and administr...,180,-,329,318,2378,1900,-,-,...,-,-,-,-,212,93,-,24,-,-
9,winter semester 2023/24,0613 Software and applications development and...,-,-,-,-,1258,-,-,-,...,-,-,-,-,-,-,-,-,-,-
10,winter semester 2023/24,073 Architecture and construction,-,36,1714,-,4319,1538,-,1286,...,-,1205,-,-,-,-,-,-,-,-


In [ ]:
universities = df.columns[2:18]
print(universities)

Index(['University of Vienna', 'University of Graz', 'University of Innsbruck',
       'University of Salzburg', 'Vienna University of Technology',
       'Graz University of Technology', 'University of Leoben',
       'University of Natural Resources and Life Sciences Vienna',
       'University of Veterinary Medicine Vienna',
       'Vienna University of Economics and Business',
       'Johannes Kepler University (JKU) Linz', 'University of Klagenfurt',
       'Danube University Krems', 'Medical University of Vienna',
       'Medical University of Graz', 'Medical University of Innsbruck'],
      dtype='object')


In [ ]:
# ceck which columns are completely empty
for column in universities:
  if df[column].empty:
    print(column, "is empty")
df["Medical University of Graz"].empty
# remove those columns
empty_columns = df.columns[(df == "-").all()]
print("Columns with only '-' values:", list(empty_columns))
df = df.drop(columns=empty_columns)

Columns with only '-' values: ['Danube University Krems', 'Medical University of Graz', 'Medical University of Innsbruck', 'Danube University Krems.1', 'Medical University of Graz.1', 'Medical University of Innsbruck.1']


In [ ]:
universities = df.columns[2:15]
print(universities)

Index(['University of Vienna', 'University of Graz', 'University of Innsbruck',
       'University of Salzburg', 'Vienna University of Technology',
       'Graz University of Technology', 'University of Leoben',
       'University of Natural Resources and Life Sciences Vienna',
       'University of Veterinary Medicine Vienna',
       'Vienna University of Economics and Business',
       'Johannes Kepler University (JKU) Linz', 'University of Klagenfurt',
       'Medical University of Vienna'],
      dtype='object')


In [ ]:
# unpivote table for normalization
df_unpivoted_male = df.melt(id_vars = ['Semester', 'Field of Study'], value_vars = universities, var_name = "University", value_name = "Number of Students")

In [ ]:
df_unpivoted_male.replace('-', 0, inplace=True)
df_unpivoted_male

<ipython-input-11-d01e59d00a30>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_unpivoted_male.replace('-', 0, inplace=True)


,Semester,Field of Study,University,Number of Students
0,winter semester 2023/24,"05 Natural sciences, mathematics and statistics",University of Vienna,6985.81994
1,winter semester 2023/24,051 Biological and related sciences,University of Vienna,2263.82000
2,winter semester 2023/24,052 Environment,University of Vienna,84.00000
3,winter semester 2023/24,053 Physical sciences,University of Vienna,3229.99994
4,winter semester 2023/24,054 Mathematics and statistics,University of Vienna,1318.00000
...,...,...,...,...
502,winter semester 2003/04,0613 Software and applications development and...,Medical University of Vienna,0.00000
503,winter semester 2003/04,073 Architecture and construction,Medical University of Vienna,0.00000
504,winter semester 2003/04,072 Manufacturing and processing,Medical University of Vienna,0.00000
505,winter semester 2003/04,071 Engineering and engineering trades,Medical University of Vienna,0.00000


In [ ]:
# create a list of universities for the female table of the second half of the table
female_universities = []

for university in universities:
  female_universities.append(university + ".1")

print(female_universities)

['University of Vienna.1', 'University of Graz.1', 'University of Innsbruck.1', 'University of Salzburg.1', 'Vienna University of Technology.1', 'Graz University of Technology.1', 'University of Leoben.1', 'University of Natural Resources and Life Sciences Vienna.1', 'University of Veterinary Medicine Vienna.1', 'Vienna University of Economics and Business.1', 'Johannes Kepler University (JKU) Linz.1', 'University of Klagenfurt.1', 'Medical University of Vienna.1']


In [ ]:
# unpivote table for normalization
df_unpivoted_female = df.melt(id_vars = ['Semester', 'Field of Study'], value_vars = female_universities, var_name = "University", value_name = "Number of Students")

In [ ]:
df_unpivoted_female.replace('-', 0, inplace = True)
df_unpivoted_female

<ipython-input-14-e6ea73e3467c>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_unpivoted_female.replace('-', 0, inplace = True)


,Semester,Field of Study,University,Number of Students
0,winter semester 2023/24,"05 Natural sciences, mathematics and statistics",University of Vienna.1,8709.11997
1,winter semester 2023/24,051 Biological and related sciences,University of Vienna.1,5054.12000
2,winter semester 2023/24,052 Environment,University of Vienna.1,160.00000
3,winter semester 2023/24,053 Physical sciences,University of Vienna.1,2451.99997
4,winter semester 2023/24,054 Mathematics and statistics,University of Vienna.1,831.00000
...,...,...,...,...
502,winter semester 2003/04,0613 Software and applications development and...,Medical University of Vienna.1,0.00000
503,winter semester 2003/04,073 Architecture and construction,Medical University of Vienna.1,0.00000
504,winter semester 2003/04,072 Manufacturing and processing,Medical University of Vienna.1,0.00000
505,winter semester 2003/04,071 Engineering and engineering trades,Medical University of Vienna.1,0.00000


In [ ]:
df_unpivoted_male.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 507 entries, 0 to 506
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Semester            507 non-null    object 
 1   Field of Study      507 non-null    object 
 2   University          507 non-null    object 
 3   Number of Students  507 non-null    float64
dtypes: float64(1), object(3)
memory usage: 16.0+ KB


In [ ]:
df_unpivoted_female.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 507 entries, 0 to 506
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Semester            507 non-null    object 
 1   Field of Study      507 non-null    object 
 2   University          507 non-null    object 
 3   Number of Students  507 non-null    float64
dtypes: float64(1), object(3)
memory usage: 16.0+ KB


In [ ]:
df_unpivoted_female.to_excel("female_students_at_public_universities_normalized.xlsx", index=False)
df_unpivoted_male.to_excel("male_students_at_public_universities_normalized.xlsx", index=False)

In [ ]:
# prepare tables for join/merge

# changing column names to assign the gender
df_unpivoted_female.rename(columns={"Number of Students" : "Number of Female Students"}, inplace = True)
df_unpivoted_male.rename(columns={"Number of Students" : "Number of Male Students"}, inplace = True)
# getting rid of the unneeded columns of the female table
df_unpivoted_female.drop(columns = ["Semester", "Field of Study", "University"], inplace = True)
df_unpivoted_female

,Number of Female Students
0,8709.11997
1,5054.12000
2,160.00000
3,2451.99997
4,831.00000
...,...
502,0.00000
503,0.00000
504,0.00000
505,0.00000


In [ ]:
df_final = df_unpivoted_male.merge(df_unpivoted_female, left_index=True, right_index=True, how = "inner")
df_final

,Semester,Field of Study,University,Number of Male Students,Number of Female Students
0,winter semester 2023/24,"05 Natural sciences, mathematics and statistics",University of Vienna,6985.81994,8709.11997
1,winter semester 2023/24,051 Biological and related sciences,University of Vienna,2263.82000,5054.12000
2,winter semester 2023/24,052 Environment,University of Vienna,84.00000,160.00000
3,winter semester 2023/24,053 Physical sciences,University of Vienna,3229.99994,2451.99997
4,winter semester 2023/24,054 Mathematics and statistics,University of Vienna,1318.00000,831.00000
...,...,...,...,...,...
502,winter semester 2003/04,0613 Software and applications development and...,Medical University of Vienna,0.00000,0.00000
503,winter semester 2003/04,073 Architecture and construction,Medical University of Vienna,0.00000,0.00000
504,winter semester 2003/04,072 Manufacturing and processing,Medical University of Vienna,0.00000,0.00000
505,winter semester 2003/04,071 Engineering and engineering trades,Medical University of Vienna,0.00000,0.00000


In [ ]:
df_final.to_excel("students_at_public_universities_normalized.xlsx", index=False)

In [ ]:
import sqlite3
# Step 1: Connect to SQLite database
db_file = "gepaus.db"
connection = sqlite3.connect(db_file)
# Step 2: Write DataFrame to database
table_name = "stem_university"
df_final.to_sql(table_name, connection, if_exists="replace", index=False)
# Step 3: Close the connection
connection.close()

print(f"DataFrame has been successfully written to {db_file}")


DataFrame has been successfully written to gepaus.db
